# Network Analysis of Car Brand Ownership

## Overview

This notebook analyzes the ownership structure of the automotive industry using a directed network.

- **Nodes** represent either parent companies or car brands
- **Directed edges** go from a parent company to each brand it owns

The goal is to identify which parent companies control the largest number of brands and to visualize the structure of automotive ownership using network analysis.

## 1. Import Libraries
This section imports the Python libraries used for data loading and network analysis.

In [3]:
import pandas as pd
import networkx as nx

## 2. Load the Dataset
The dataset contains car brands, their parent companies, and the Wikipedia source used for manual data collection.

In [4]:
df = pd.read_csv("car_brands_parent_companies.csv")

In [5]:
df.head()

,brand,parent_company,source_URL,Unnamed: 3
0,Volkswagen,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group,NaN
1,Audi,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group,NaN
2,Porsche,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group,NaN
3,Lamborghini,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group,NaN
4,Bentley,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group,NaN


## 3. Clean the Dataset
Excel exported an extra unnamed column, so it is removed here.

In [6]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [7]:
df.head()

,brand,parent_company,source_URL
0,Volkswagen,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group
1,Audi,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group
2,Porsche,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group
3,Lamborghini,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group
4,Bentley,Volkswagen Group,https://en.wikipedia.org/wiki/Volkswagen_Group


## 4. Build the Network Graph

This graph is **directed**.

- A **parent company** is represented as a node
- A **car brand** is represented as a node
- A **directed edge** is created from a parent company to each brand it owns

This creates a parent-to-brand ownership network instead of connecting brands to each other.

In [8]:
G = nx.DiGraph()

# Add parent company nodes
for parent in df["parent_company"].unique():
    G.add_node(parent, node_type="parent")

# Add brand nodes
for brand in df["brand"].unique():
    G.add_node(brand, node_type="brand")

# Add directed edges: parent company -> brand
for _, row in df.iterrows():
    G.add_edge(row["parent_company"], row["brand"])

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 50
Number of edges: 40


## 5. Out-Degree Analysis
Out-degree measures how many outgoing connections a node has.

In this ownership network, parent companies with higher out-degree own more brands, so out-degree is used here as the main measure of importance.

In [9]:
out_degree = dict(G.out_degree())

sorted_out_degree = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)

print("Top 10 nodes by out-degree:")
for node, value in sorted_out_degree[:10]:
    print(node, value)

Top 10 nodes by out-degree:
Stellantis 12
Volkswagen Group 8
General Motors 4
BMW Group 3
Hyundai Motor Company 3
Toyota Motor Corporation 2
Mercedes-Benz Group 2
Ford Motor Company 2
Honda Motor Company 2
Nissan Motor Company 2


## 6. Betweenness Centrality

Betweenness centrality measures how often a node appears on shortest paths between other nodes.

In this ownership network, betweenness centrality is less informative than out-degree because the graph is structured as a set of parent companies pointing directly to the brands they own. As a result, parent companies may appear central, while brands usually have very low betweenness.

In [10]:
betweenness = nx.betweenness_centrality(G)
sorted_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)

print("Top 10 nodes by betweenness centrality:")
for node, value in sorted_between[:10]:
    print(node, round(value, 4))

Top 10 nodes by betweenness centrality:
Volkswagen Group 0.0
BMW Group 0.0
Toyota Motor Corporation 0.0
Mercedes-Benz Group 0.0
General Motors 0.0
Ford Motor Company 0.0
Stellantis 0.0
Honda Motor Company 0.0
Nissan Motor Company 0.0
Hyundai Motor Company 0.0


## 7. Export Graph for Gephi

The graph is exported as a `.gexf` file so it can be opened and visualized in Gephi.

In [11]:
nx.write_gexf(G, "car_company_parent_brand_network.gexf")

## 8. Summary of Top Nodes
This table summarizes the nodes with the highest out-degree values in the ownership network.

In [12]:
top_out_degree_df = pd.DataFrame(sorted_out_degree[:10], columns=["node", "out_degree"])
top_out_degree_df

,node,out_degree
0,Stellantis,12
1,Volkswagen Group,8
2,General Motors,4
3,BMW Group,3
4,Hyundai Motor Company,3
5,Toyota Motor Corporation,2
6,Mercedes-Benz Group,2
7,Ford Motor Company,2
8,Honda Motor Company,2
9,Nissan Motor Company,2


In [13]:
top_betweenness_df = pd.DataFrame(sorted_between[:10], columns=['brand', 'betweenness_centrality'])
top_betweenness_df

,brand,betweenness_centrality
0,Volkswagen Group,0.0
1,BMW Group,0.0
2,Toyota Motor Corporation,0.0
3,Mercedes-Benz Group,0.0
4,General Motors,0.0
5,Ford Motor Company,0.0
6,Stellantis,0.0
7,Honda Motor Company,0.0
8,Nissan Motor Company,0.0
9,Hyundai Motor Company,0.0


## 9. Conclusion

The network analysis shows that a small number of parent companies control many car brands in the global automotive industry. Using a directed parent-to-brand ownership network makes these ownership relationships easier to interpret than a brand-to-brand network.

Out-degree was used as the main measure of importance, since parent companies with more outgoing edges own more brands. Based on this structure, Stellantis and Volkswagen Group emerge as the most important nodes because they control the largest number of brands in the dataset.

This project demonstrates how network analysis can be used to visualize and analyze corporate ownership structures in a way that is more intuitive than traditional tables or lists.